In [3]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader
from analysis import fit_peak_expbg

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used
# Energy emission peaks in MeV
K40_MEV = [1.460]
TL208_MEV = [2.614]
CS137_MEV = [0.661]
ANNIHIL_MEV = 0.511
NA22_MEV = [ANNIHIL_MEV, 1.2745, 1.2745+ANNIHIL_MEV]
CO60_MEV = [(1.1732+1.3325)/2, 1.1732+1.3325] # First value is the average of the two photopeaks. They have the same branching ratio

# colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen, ROOT.kOrange, ROOT.kAzure, ROOT.kSpring, ROOT.kPink, ROOT.kMagenta, ROOT.kTeal, ROOT.kYellow, ROOT.kViolet, ROOT.kCyan]
# colors = [ROOT.kAzure+9, ROOT.kOrange+1, ROOT.kRed+1, ROOT.kGray+1, ROOT.kViolet-5, ROOT.kRed-5, ROOT.kOrange+8, ROOT.kYellow-5, ROOT.kGray+2, ROOT.kCyan-8]
colors = [ROOT.kAzure-7, ROOT.kOrange+1, ROOT.kRed-7, ROOT.kCyan-5, ROOT.kGreen-5, ROOT.kOrange-4, ROOT.kMagenta-5, ROOT.kRed-9, ROOT.kOrange-7, ROOT.kGray+1]

In [6]:
folder = 'data/DataForOscar_260608/'
files_A = sorted((project_root/folder).glob('SUBT_A[1-4]*.csv'))
files_B = sorted((project_root/folder).glob('SUBT_B[1-4]*.csv'))
print(files_A)
print(files_B)


[PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/DataForOscar_260608/SUBT_A1_BackgroundAtStart_DoubleSiphra_DisableCh15SiphraB.csv'), PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/DataForOscar_260608/SUBT_A2_Cs137OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB.csv'), PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/DataForOscar_260608/SUBT_A3_Co60OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB.csv'), PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/DataForOscar_260608/SUBT_A4_Na22OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB.csv')]
[PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/DataForOscar_260608/SUBT_B1_BackgroundAtStart_DoubleSiphra_DisableCh15SiphraB.csv'), PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/DataForOscar_260608/SUBT_B2_Cs137OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB.csv'), PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/DataForOscar_260608/SUBT_B3_Co60OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB.csv'), PosixPath('/home/

In [7]:
acqs_A = [SiphraAcquisition(file) for file in files_A]
acqs_B = [SiphraAcquisition(file) for file in files_B]
print(acqs_A)
print(acqs_B)

[SIPHRAAcquisition(File: 'SUBT_A1_BackgroundAtStart_DoubleSiphra_DisableCh15SiphraB'), SIPHRAAcquisition(File: 'SUBT_A2_Cs137OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB'), SIPHRAAcquisition(File: 'SUBT_A3_Co60OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB'), SIPHRAAcquisition(File: 'SUBT_A4_Na22OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB')]
[SIPHRAAcquisition(File: 'SUBT_B1_BackgroundAtStart_DoubleSiphra_DisableCh15SiphraB'), SIPHRAAcquisition(File: 'SUBT_B2_Cs137OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB'), SIPHRAAcquisition(File: 'SUBT_B3_Co60OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB'), SIPHRAAcquisition(File: 'SUBT_B4_Na22OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB')]


In [8]:
# histograms
from collections import namedtuple

DataHandler = namedtuple('DataHandler', ['acq', 'hist', 'src', 'subt'])

hists = {}
sources = []
handlers = {}
for acq, crystal in zip([*acqs_A, *acqs_B], [*('A'*4), *('B'*4)]):
    filepath = acq.filepath.stem
    src = (MetadataLoader.load(acq.metadataFile)).source
    src_key = 'BG' if src == 'Background' else src[:2]
    sources.append(src)
    siphra_code = acq.filepath.stem[5]
    print(f'{src} - Detector {crystal}')
    # Generate histograms
    hist = ROOT.TH1F(f"{src}_SIPHRA_{siphra_code}", "", BITS12, 0, BITS12)
    hist.Fill(acq['s']/len(acq.active_chs))
    hists[f'{crystal}_{src_key}'] = hist
    handlers[f'{crystal}_{src_key}'] = DataHandler(acq, hist, src, None)
    del hist

Background - Detector A
Cs-137 - Detector A
Co-60 - Detector A
Na-22 - Detector A
Background - Detector B
Cs-137 - Detector B
Co-60 - Detector B
Na-22 - Detector B


In [60]:
# Subtract backgrounds here if correction factors are not configured, checking each one

correction_factors = {
    'A_Cs': 1.3,
    'A_Na': 1.5,
    'A_Co': 1.7,
    'B_Cs': 1.3,
    'B_Na': 1.7,
    'B_Co': 1.6,
}

if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 800, 600)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgd = ROOT.TLegend(0.48, 0.61, 0.75, 0.83)

whichSiphra = 'B'
source = 'Na'
bg = handlers[f'{whichSiphra}_BG']
data_key = f'{whichSiphra}_{source}'
sgnl = handlers[data_key]

bg_scaled = bg.hist.Clone()
correction_factor = correction_factors[data_key] if correction_factors[data_key] else 1
bg_scaled.Scale(correction_factor*sgnl.acq.exposure/bg.acq.exposure)

subtracted = sgnl.hist.Clone(f'{sgnl.src}_{whichSiphra}_Subt.')
subtracted.Add(bg_scaled, -1)
# subts[data_key].hist = subtracted

lgd.AddEntry(sgnl.hist, 'Signal')
lgd.AddEntry(bg_scaled, 'Background')
lgd.AddEntry(subtracted, 'Subtracted')
sgnl.hist.SetLineColor(colors[5])
bg_scaled.SetLineColor(colors[3])
subtracted.SetLineColor(colors[2])
sgnl.hist.SetTitle(f'{sgnl.src} SIPHRA {data_key[:1]}')
bg_scaled.Draw('hist')
sgnl.hist.Draw('hist sames')
subtracted.Draw('hist sames')
cv.SetLogy()
lgd.Draw()
cv.Draw()

new_handler = DataHandler(sgnl.acq, sgnl.hist, sgnl.src, subtracted)
handlers[data_key] = new_handler

In [34]:
handlers

{'A_BG': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_A1_BackgroundAtStart_DoubleSiphra_DisableCh15SiphraB'), hist=<cppyy.gbl.TH1F object at 0x5e00be5b90c0>, src='Background', subt=None),
 'A_Cs': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_A2_Cs137OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB'), hist=<cppyy.gbl.TH1F object at 0x5e00beaa3a20>, src='Cs-137', subt=<cppyy.gbl.TH1F object at 0x5e00c1af14e0>),
 'A_Co': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_A3_Co60OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB'), hist=<cppyy.gbl.TH1F object at 0x5e00be412fd0>, src='Co-60', subt=<cppyy.gbl.TH1F object at 0x5e00c0bdbfb0>),
 'A_Na': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_A4_Na22OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB'), hist=<cppyy.gbl.TH1F object at 0x5e00be4a6450>, src='Na-22', subt=<cppyy.gbl.TH1F object at 0x5e00b99bfbe0>),
 'B_BG': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_B1_BackgroundAtStart_DoubleSiphra_DisableCh15SiphraB'), hist=<cppyy.gbl.TH1F ob

In [63]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

Yinit = 0.82 # For stat boxes

cv = ROOT.TCanvas('cv', 'cv', 1600, 600)
cv.Divide(2,1)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgds = [ROOT.TLegend(0.48, 0.61, 0.75, 0.83) for _ in range(2)]

srcs_list = list(handlers.values())
srcs_list.pop(4)
srcs_list.pop(0)


for n, handler in enumerate(srcs_list):
    cv.cd(n//3 + 1)
    hist = handler.subt
    lgds[n//3].AddEntry(hist, handler.src)
    if not n%3:
        s = 'A' if n < 3 else 'B'
        hist.SetTitle(f'Detector {s}')
        hist.GetXaxis().SetTitle("ADC channel number")
        hist.GetYaxis().SetTitle("Relative counts")
        hist.SetLineColor(colors[(n%3)])
        hist.Draw('hist')
    else:
        hist.SetLineColor(colors[(n%3)])
        hist.Draw('hist sames')

for c in range(2):
    cv.cd(c+1)
    lgds[c].Draw()
    cv.cd(c+1).SetLogy()

cv.Draw()


In [62]:
# Fit ranges for each peak. Same order as in srcs_list
fit_ranges = {
    'A_Cs': [(227,418)], # Cs siphra A
    'A_Na': [(168,335),(505,634),(689,870)], # Na siphra A
    'A_Co': [(466,634),(995,1195)], # Co siphra A
    'B_Cs': [(192,385)],
    'B_Na': [(142,305),(440,572),(642,816)],
    'B_Co': [(424,649),(894, 1109)]
}

for k, ranges in fit_ranges.items():
    try:
        hist = handlers[k].subt
        source = handlers[k].src
        chip = k[0]
        # print(f'{k}: chip: {chip}, src: {source}')
    except:
        print(f'Unmatched key {k}')

    for n, r in enumerate(ranges):
        fit_fn = ROOT.TF1(f"{source}_gaus_{n+1}", "gaus", 3)
        fit_status = hist.Fit(fit_fn, 'L S +', "", r[0], r[1])

****************************************
Minimizer is Minuit2 / Migrad
MinFCN                    =      797.728
Chi2                      =      1595.46
NDf                       =          188
Edm                       =  2.75013e-06
NCalls                    =           66
Constant                  =      2525.61   +/-   6.67341     
Mean                      =      314.395   +/-   0.0796145   
Sigma                     =      36.4852   +/-   0.0637769    	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
MinFCN                    =          166
Chi2                      =      332.001
NDf                       =          164
Edm                       =  3.73023e-08
NCalls                    =           72
Constant                  =      251.158   +/-   2.13246     
Mean                      =      245.392   +/-   0.275909    
Sigma                     =      38.1447   +/-   0.248234     	 (limited)
****************************************
Minimizer i

In [18]:
handlers['A_Co'].subt.GetListOfFunctions().ls()

OBJ: TList	TList	Doubly linked list : 0
 OBJ: TPaveStats	stats  	X1= 3686.400140 Y1=3.632541 X2=4505.600134 Y2=4.022536
 OBJ: TF1	Co-60_gaus_1	gaus : 0 at: 0x563dae858840
 OBJ: TF1	Co-60_gaus_2	gaus : 0 at: 0x563db16f4720


In [64]:
# graph of position of photopeaks

def get_mean_and_error(func_name, hist):
    f = hist.GetFunction(func_name)
    idx = f.GetParNumber('Mean')
    return f.GetParameter(idx), f.GetParError(idx)

COLOR_A = colors[0]
COLOR_B = colors[1]

means_A, errors_A = zip(*[
    get_mean_and_error('Cs-137_gaus_1', handlers['A_Cs'].subt),
    get_mean_and_error('Na-22_gaus_1',  handlers['A_Na'].subt),
    get_mean_and_error('Na-22_gaus_2',  handlers['A_Na'].subt),
    get_mean_and_error('Na-22_gaus_3',  handlers['A_Na'].subt),
    get_mean_and_error('Co-60_gaus_1',  handlers['A_Co'].subt),
    get_mean_and_error('Co-60_gaus_2',  handlers['A_Co'].subt),
])

means_A  = np.array(means_A,  dtype=np.float64)
errors_A = np.array(errors_A, dtype=np.float64)

means_B, errors_B = zip(*[
    get_mean_and_error('Cs-137_gaus_1', handlers['B_Cs'].subt),
    get_mean_and_error('Na-22_gaus_1',  handlers['B_Na'].subt),
    get_mean_and_error('Na-22_gaus_2',  handlers['B_Na'].subt),
    get_mean_and_error('Na-22_gaus_3',  handlers['B_Na'].subt),
    get_mean_and_error('Co-60_gaus_1',  handlers['B_Co'].subt),
    get_mean_and_error('Co-60_gaus_2',  handlers['B_Co'].subt),
])

means_B  = np.array(means_B,  dtype=np.float64)
errors_B = np.array(errors_A, dtype=np.float64)

peaks_real = np.array([*CS137_MEV, *NA22_MEV, *CO60_MEV])

if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 800, 600)
lg = ROOT.TLegend(0.48, 0.61, 0.75, 0.83)

zeros = np.zeros(len(means_A), dtype=np.float64)

peaks_graph_A = ROOT.TGraph(6, means_A, peaks_real)
peaks_graph_A.SetMarkerColor(COLOR_A)
peaks_graph_A.SetMarkerStyle(20)
correction_B = 1
peaks_graph_B = ROOT.TGraph(6, means_B*correction_B, peaks_real)
peaks_graph_B.SetMarkerColor(COLOR_B)
peaks_graph_B.SetMarkerStyle(20)

peaks_graph_A.SetTitle("Position of the photopeaks of the radiation sources")
peaks_graph_A.GetXaxis().SetTitle("ADC channel number")
peaks_graph_A.GetYaxis().SetTitle("Energy (MeV)")

lg.AddEntry(peaks_graph_A, "SIPHRA A")
lg.AddEntry(peaks_graph_B, "SIPHRA B")
peaks_graph_A.Draw("AP")
peaks_graph_B.Draw("P same")
lg.Draw()
cv.Draw()

In [66]:
# Fitting the calibration curves

calib_A = ROOT.TF1('Calibration_A', 'pol1', means_A.min()-200, means_A.max()+200)
peaks_graph_A.Fit(calib_A, 'RS 0')

calib_B = ROOT.TF1('Calibration_B', 'pol1', means_B.min()-200, means_B.max()+200)
peaks_graph_B.Fit(calib_B, 'RS 0')



****************************************
Minimizer is Linear / Migrad
Chi2                      =   0.00239873
NDf                       =            4
p0                        =   -0.0681398   +/-   0.0230836   
p1                        =   0.00237293   +/-   3.52683e-05 
****************************************
Minimizer is Linear / Migrad
Chi2                      =    0.0032224
NDf                       =            4
p0                        =   -0.0404799   +/-   0.0263294   
p1                        =   0.00253677   +/-   4.37066e-05 


In [67]:
#### if ROOT.gROOT.FindObject('cv'):
ROOT.gROOT.FindObject('cv').Close()

labels = [
    "{}^{137}Cs " + f"{CS137_MEV[0]:.2f} MeV",
    "{}^{22}Na " + f"{NA22_MEV[0]:.2f} MeV",
    "{}^{22}Na " + f"{NA22_MEV[1]:.2f} MeV",
    "{}^{22}Na " + f"{NA22_MEV[2]:.2f} MeV",
    "{}^{60}Co " + f"{CO60_MEV[0]:.2f} MeV",
    "{}^{60}Co " + f"{CO60_MEV[1]:.2f} MeV",
    "{}^{40}K "  + f"{K40_MEV[0]:.2f} MeV",
    "{}^{208}Tl "+ f"{TL208_MEV[0]:.2f} MeV",
          ]

real_peaks_ordered = [*CS137_MEV, *NA22_MEV, *CO60_MEV, *K40_MEV, *TL208_MEV]


cv = ROOT.TCanvas('cv', 'cv', 1200, 800)
lg = ROOT.TLegend(0.15, 0.75, 0.36, 0.88)

lg.AddEntry(peaks_graph_A, "Detector A")
lg.AddEntry(peaks_graph_B, "Detector B")

calib_A.SetLineColor(COLOR_A)
calib_A.SetLineStyle(2)
calib_B.SetLineColor(COLOR_B)
calib_B.SetLineStyle(2)

peaks_graph_A.Draw("AP")
peaks_graph_B.Draw("P same")
calib_A.Draw("same")
calib_B.Draw("same")
lg.Draw()

# Fit parameters

# eqbox_A = ROOT.TPaveText(0.15, 0.65, 0.45, 0.85, "NDC")
# eqbox_A.SetFIllColor(0)
# eqbox_A.(1)

# get parameters
calibPars = {}
eqboxes = {}
for chip, fitfn in zip(['A', 'B'], [calib_A, calib_B]):
    eqboxes[f'{chip}'] = ROOT.TPaveText(0.15, 0.7, 0.48, 0.8, "NDC")
    box =  eqboxes[f'{chip}']
    box.SetFillColor(0)
    box.SetBorderSize(1)
    a = fitfn.GetParameter(0)
    b = fitfn.GetParameter(1)
    sa = fitfn.GetParError(0)
    sb = fitfn.GetParError(1)

    calibPars[f'a_{chip}'] = a
    calibPars[f'b_{chip}'] = b
    calibPars[f'sa_{chip}'] = sa
    calibPars[f'sb_{chip}'] = sb

    box.AddText(f"Detector {chip}")
    box.AddText(f"E = ({a:.4f} #pm {sa:.4f}) + ({b:.4f} #pm {sb:.4f}) x")
    box.Draw()

# Add labels
latex = ROOT.TLatex()
latex.SetTextSize(0.02)
latex.SetTextAlign(12)  # alineación: horizontal izquierda, vertical centrada

for i, adc in enumerate(means_A):
    latex.DrawLatex(adc+ 0.05, real_peaks_ordered[i], labels[i])  # pequeño offset en x


cv.Update()
cv.Draw()

# Dynamic range and verification

In [171]:
# Calibrate BG spectra

def calibrate(data, detector, return_errs=False):
    ''' Performs linear energy calibration (a + bx) over a data array containing ADC values.'''
    a = calibPars[f'a_{detector}']
    b = calibPars[f'b_{detector}']
    sa = calibPars[f'sa_{detector}']
    sb = calibPars[f'sb_{detector}']
    value = a + b*data
    if return_errs:
        dva = 1
        dvb = b
        err = np.sqrt((dva*sa)**2 + (dvb*sb)**2)
        return a + b*data, err
    return value

data_bgA_calib = calibrate(handlers['A_BG'].acq['s']/16, detector='A')
data_bgB_calib = calibrate(handlers['B_BG'].acq['s']/16, detector='B')

hist_bgA_calib = ROOT.TH1F(f"BG_SIPHRA_A", "", BITS12, data_bgA_calib.min(), data_bgA_calib.max())
hist_bgA_calib.Fill(data_bgA_calib)
hist_bgB_calib = ROOT.TH1F(f"BG_SIPHRA_B", "", BITS12, data_bgB_calib.min(), data_bgB_calib.max())
hist_bgB_calib.Fill(data_bgB_calib)

Warning in <TROOT::Append>: Replacing existing TH1: BG_SIPHRA_A (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: BG_SIPHRA_B (Potential memory leak).


In [113]:
print('Dynamic ranges:')
det_ids = ('A', 'B')
min_bins = (-calibPars['a_A']/calibPars['b_A'], -calibPars['a_B']/calibPars['b_B'])
pedestal_maxs = (108, 116)
pedestal_energies = (calibrate(pedestal_maxs[0], detector='A', return_errs=True), calibrate(pedestal_maxs[1], detector='B', return_errs=True))
dynRange_min = (calibrate(min_bins[0], detector='A', return_errs=True), calibrate(min_bins[1], detector='B', return_errs=True))
dynRange_max = (calibrate(BITS12, detector='A', return_errs=True), calibrate(BITS12, detector='B', return_errs=True))

print(dynRange_max)

print(f'{'-'*80}')
print(f'{' '*15} | {'L bound':^16} | {'R bound':^16} | {'Pedestal R':^16}')
print(f'{'-'*80}')
print(f'{'Detector A':^15} | {f'{dynRange_min[0][0]:.3f} ± {dynRange_min[0][1]:3f}':^16} | {f'{dynRange_max[0][0]:.4f} ± {dynRange_max[0][1]:.4f}':^16} | {pedestal_energies[0][0]:.4f} ± {pedestal_energies[0][1]:.4f}')
print(f'{'Detector B':^15} | {dynRange_min[1][0]:.3f} ± {dynRange_min[1][1]:3f} | {dynRange_max[1][0]:.4f} ± {dynRange_max[1][1]:.4f}| {pedestal_energies[1][0]:.4f} ± {pedestal_energies[1][1]:.4f}')

Dynamic ranges:
((9.65136210908262, np.float64(0.02308361458875107)), (10.350137966294605, np.float64(0.026329449922635813)))
--------------------------------------------------------------------------------
                |     L bound      |     R bound      |    Pedestal R   
--------------------------------------------------------------------------------
  Detector A    | 0.000 ± 0.023084 | 9.6514 ± 0.0231  | 0.1881 ± 0.0231
  Detector B    | 0.000 ± 0.026329 | 10.3501 ± 0.0263| 0.2538 ± 0.0263


In [174]:
import visualization

visualization.hist_quickShow([hist_bgA_calib, hist_bgB_calib], legends=['Detector A','Detector B'], color_offset=5, tone_offset=-1)

(<cppyy.gbl.TCanvas object at 0x5e00c578fb00>,
 <cppyy.gbl.TLegend object at 0x5e00c57bb210>)

In [173]:
import importlib
if 'analysis' in sys.modules:
    del sys.modules['analysis']

from analysis import fit_peak_expbg

print(f'{K40_MEV[0]=}    {TL208_MEV[0]=}')

fit_peak_expbg(hist_bgA_calib, 'K_peak', 1.3, 1.6, 40, 1.46, 0.3, showFit=True, keep_prev_fncs=True)
fit_peak_expbg(hist_bgA_calib, 'Tl_peak', 2.48, 2.9, 5, 2.61, 0.3, showFit=True, keep_prev_fncs=True)

fit_peak_expbg(hist_bgB_calib, 'K_peak', 1.26, 1.52, 20, 1.46, 0.3, showFit=True, keep_prev_fncs=True)
fit_peak_expbg(hist_bgB_calib, 'Tl_peak', 2.4, 3, 5, 2.61, 0.3, showFit=True, keep_prev_fncs=True)

K40_MEV[0]=1.46    TL208_MEV[0]=2.614


(<cppyy.gbl.TF1 object at 0x5e00c57a3070>,
 <cppyy.gbl.TFitResultPtr object at 0x5e00c57b5490>)

****************************************
Minimizer is Minuit2 / Migrad
MinFCN                    =      66.0962
Chi2                      =      132.192
NDf                       =          127
Edm                       =  4.58087e-06
NCalls                    =          776
Const                     =      10.6187   +/-   2.04425     
Decay                     =     -7.12808   +/-   1.60924     
Norm                      =      15.7735   +/-   1.47822     
Mean                      =      1.44819   +/-   0.014797    
Sigma                     =     0.116227   +/-   0.0147743   
****************************************
Minimizer is Minuit2 / Migrad
MinFCN                    =      108.538
Chi2                      =      217.076
NDf                       =          180
Edm                       =  8.87982e-08
NCalls                    =          883
Const                     =      3.87483   +/-   1.21619     
Decay                     =     -1.53914   +/-   0.452322    
Norm          

In [175]:
bgHists = [hist_bgA_calib, hist_bgB_calib]
bgPeaks_real = [K40_MEV, TL208_MEV]
kpeaks = [bgHists[0].GetFunction('K_peak').GetParameter('Mean'), bgHists[1].GetFunction('K_peak').GetParameter('Mean')]
tlpeaks = [bgHists[0].GetFunction('Tl_peak').GetParameter('Mean'), bgHists[1].GetFunction('Tl_peak').GetParameter('Mean')]
errs = []
errs = [[(expr - real[0])/real[0] for expr in l_exp] for real, l_exp in zip(bgPeaks_real, [kpeaks, tlpeaks])]
print("Errors in the energy reconstruction of the background K-40 and Tl-208 photopeaks")
print(f'{'-'*50}')
print(f'{'Source':^10} | {'SIPHRA A':^15} | {'SIPHRA B':^15}')
print(f'{'-'*50}')
print(f'{'K-40':^10} | {f'{errs[0][0]*100:.3f} %':>15} | {f'{errs[0][1]*100:.3f} %':>15}')
print(f'{'Tl-208':^10} | {f'{errs[1][0]*100:.3f} %':>15} | {f'{errs[1][1]*100:.3f} %':>15}')
print(f'{'-'*50}')

Errors in the energy reconstruction of the background K-40 and Tl-208 photopeaks
--------------------------------------------------
  Source   |    SIPHRA A     |    SIPHRA B    
--------------------------------------------------
   K-40    |        -0.809 % |        -3.247 %
  Tl-208   |         2.372 % |        -5.876 %
--------------------------------------------------


# Energy resolution

For the energy resolution, we can get the calibrated data from the fitted data of the spectra we previously obtained. For a linear calibration $E = a + b \cdot x$, the transform for the parameters of the fitted gaussians is:

**Mean**

$\mu_E = a + b\cdot \mu_x$

**Sigma**

$\sigma_E = b \cdot \sigma_x$

The propagated error in the sigma is

$\sigma^2_{\sigma_E} = b^2\sigma^2_{\sigma_x} + \sigma^2_x\sigma^2_b + 2b\sigma_x \mathrm{cov}(\sigma_x, b)$

The last term is negligible since $\sigma_x$ comes from the histogram fit and $b$ from the calibration, so they are independent.

In [176]:
def get_sigma_and_error(func_name, hist):
    f = hist.GetFunction(func_name)
    idx = f.GetParNumber('Sigma')
    return f.GetParameter(idx), f.GetParError(idx)

COLOR_A = colors[0]
COLOR_B = colors[1]



sigmas_A, errs_sig_A = zip(*[
    get_sigma_and_error('Cs-137_gaus_1', handlers['A_Cs'].subt),
    get_sigma_and_error('Na-22_gaus_1',  handlers['A_Na'].subt),
    get_sigma_and_error('Na-22_gaus_2',  handlers['A_Na'].subt),
    get_sigma_and_error('Na-22_gaus_3',  handlers['A_Na'].subt),
    get_sigma_and_error('Co-60_gaus_1',  handlers['A_Co'].subt),
    get_sigma_and_error('Co-60_gaus_2',  handlers['A_Co'].subt),
])

sigmas_B, errs_sig_B = zip(*[
    get_sigma_and_error('Cs-137_gaus_1', handlers['B_Cs'].subt),
    get_sigma_and_error('Na-22_gaus_1',  handlers['B_Na'].subt),
    get_sigma_and_error('Na-22_gaus_2',  handlers['B_Na'].subt),
    get_sigma_and_error('Na-22_gaus_3',  handlers['B_Na'].subt),
    get_sigma_and_error('Co-60_gaus_1',  handlers['B_Co'].subt),
    get_sigma_and_error('Co-60_gaus_2',  handlers['B_Co'].subt),
])

PeakPars = namedtuple('PeakPars', ['mean', 'sigma', 'err_mean', 'err_sigma', 'fwhm', 'res'])

fwhm_factor = np.sqrt(8*np.log(2))
src_peaks_pars = {}
peak_names = ['Cs1', 'Na1', 'Na2', 'Na3', 'Co1', 'Co2']
real_peaks_ordered = [*CS137_MEV, *NA22_MEV, *CO60_MEV, *K40_MEV, *TL208_MEV]

for chip, means, err_means, sigs, err_sigs, a, b, err_b in zip(('A', 'B'), (means_A, means_B), (errors_A, errors_B), (sigmas_A, sigmas_B), (errs_sig_A, errs_sig_B), (calibPars['a_A'], calibPars['a_B']), (calibPars['b_A'], calibPars['b_B']), (calibPars['sb_A'], calibPars['sb_B'])):
    # print(chip, means, err_means, sigs, err_sigs, a, b, err_b, '\n')
    for name, m, sm, s, ss, r in zip((peak_names), means, err_means, sigs, err_sigs, real_peaks_ordered):
        mean, err_mean = a + b*m, np.sqrt((b*sm)**2 + (m*err_b)**2)
        sig, err_sig = b*s, np.sqrt((b*ss)**2 + (s*err_b)**2)
        fwhm = fwhm_factor*sig
        res = fwhm/mean
        src_peaks_pars[f'{chip}_{name}'] = PeakPars(mean=mean, sigma=sig, err_mean=err_mean, err_sigma=err_sig, fwhm=fwhm, res=res*100)

for k, v in src_peaks_pars.items():
    print(f'{k}:\n {v}\n')

A_Cs1:
 PeakPars(mean=np.float64(0.6778953126863144), sigma=0.08657674818620299, err_mean=np.float64(0.011089776398036901), err_sigma=np.float64(0.0012956414636319372), fwhm=np.float64(0.20387266206246768), res=np.float64(30.074357831827434))

A_Na1:
 PeakPars(mean=np.float64(0.5141564134750306), sigma=0.09051464056953853, err_mean=np.float64(0.008679278159084598), err_sigma=np.float64(0.0014686057097793505), fwhm=np.float64(0.21314568998192093), res=np.float64(41.45541792259917))

A_Na2:
 PeakPars(mean=np.float64(1.2782435933598293), sigma=0.09319990995813923, err_mean=np.float64(0.020078783095139063), err_sigma=np.float64(0.0023621215320194207), fwhm=np.float64(0.21946901616450584), res=np.float64(17.16957685566312))

A_Na3:
 PeakPars(mean=np.float64(1.7478363355378888), sigma=0.12208950108062594, err_mean=np.float64(0.02705622771449042), err_sigma=np.float64(0.0026619492370064455), fwhm=np.float64(0.2874988044324857), res=np.float64(16.448840122322395))

A_Co1:
 PeakPars(mean=np.flo

In [177]:
resolutions = {'A':[], 'B':[]}

for k, v in src_peaks_pars.items():
    resolutions[k[0]].append(v.res)

# Add the background peaks data
real_peaks_bg = [K40_MEV[0], TL208_MEV[0]]
res_bg = {'A': [hist_bgA_calib.GetFunction(fnc).GetParameter('Sigma') for fnc in ['K_peak', 'Tl_peak']],
         'B': [hist_bgB_calib.GetFunction(fnc).GetParameter('Sigma') for fnc in ['K_peak', 'Tl_peak']]}
# print(res_bg)

means_bg = {'A':[hist_bgA_calib.GetFunction(fnc).GetParameter('Mean') for fnc in ['K_peak', 'Tl_peak']], 
            'B':[hist_bgB_calib.GetFunction(fnc).GetParameter('Mean') for fnc in ['K_peak', 'Tl_peak']]}

for k, v in res_bg.items():
    # print(k, v)
    res_bg[k] = [fwhm_factor*100*sig/r for sig, r in zip(v, means_bg[k])]

resolutions['A'].extend(res_bg['A'])
resolutions['B'].extend(res_bg['B'])

print(f'{resolutions=}')

resolutions={'A': [np.float64(30.074357831827434), np.float64(41.45541792259917), np.float64(17.16957685566312), np.float64(16.448840122322395), np.float64(22.934866035575126), np.float64(13.934713280264852), np.float64(18.898944707980114), np.float64(7.223292743076707)], 'B': [np.float64(37.373323117662515), np.float64(48.76914586326189), np.float64(23.45939702404722), np.float64(16.761810340878334), np.float64(25.64842272836114), np.float64(16.138665443228266), np.float64(15.580676724984064), np.float64(14.558459792354467)]}


In [29]:

labels = [
          f"{{}}^{137}Cs {CS137_MEV[0]} ",
          ]

labels

['{}^137Cs 0.661 ']

In [178]:
real_peaks_ordered = np.array(real_peaks_ordered)
resolutions['A'] = np.array(resolutions['A'])
resolutions['B'] = np.array(resolutions['B'])

labels = [
    "{}^{137}Cs " + f"{CS137_MEV[0]:.2f} MeV",
    "{}^{22}Na " + f"{NA22_MEV[0]:.2f} MeV",
    "{}^{22}Na " + f"{NA22_MEV[1]:.2f} MeV",
    "{}^{22}Na " + f"{NA22_MEV[2]:.2f} MeV",
    "{}^{60}Co " + f"{CO60_MEV[0]:.2f} MeV",
    "{}^{60}Co " + f"{CO60_MEV[1]:.2f} MeV",
    "{}^{40}K "  + f"{K40_MEV[0]:.2f} MeV",
    "{}^{208}Tl "+ f"{TL208_MEV[0]:.2f} MeV",
          ]

if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 800, 600)
lg = ROOT.TLegend(0.48, 0.61, 0.75, 0.83)

zeros = np.zeros(len(means_A), dtype=np.float64)

res_graph_A = ROOT.TGraph(8, real_peaks_ordered, resolutions['A'])
res_graph_A.SetMarkerColor(COLOR_A)
res_graph_A.SetMarkerStyle(20)
correction_B = 1
res_graph_B = ROOT.TGraph(8, real_peaks_ordered, resolutions['B'])
res_graph_B.SetMarkerColor(COLOR_B)
res_graph_B.SetMarkerStyle(20)

res_graph_A.SetTitle("Energy resolution")
res_graph_A.GetXaxis().SetTitle("Energy (MeV)")
res_graph_A.GetYaxis().SetTitle("Resolution (%)")

lg.AddEntry(res_graph_A, "Detector A")
lg.AddEntry(res_graph_B, "Detector B")
res_graph_A.Draw("AP")
res_graph_B.Draw("P same")

# Add labels
latex = ROOT.TLatex()
latex.SetTextSize(0.02)
latex.SetTextAlign(12)  # alineación: horizontal izquierda, vertical centrada

for i in range(len(resolutions['A'])):
    latex.DrawLatex(real_peaks_ordered[i] + 0.05, resolutions['A'][i], labels[i])  # pequeño offset en x


lg.Draw()
cv.Update()
cv.Draw()

In [179]:
res_trend_A = ROOT.TF1('res_trend_A', 'TMath::Sqrt([0]^2 + [1]^2*x + [2]^2*x^2)/x', 0, 3)
# res_trend_A.FixParameter(0, 0)
res_graph_A.Fit(res_trend_A, 'RS 0')
res_trend_B = ROOT.TF1('res_trend_B', 'TMath::Sqrt([0]^2 + [1]^2*x + [2]^2*x^2)/x', 0, 3)
# res_trend_B.FixParameter(0, 0)
res_graph_B.Fit(res_trend_B, 'RS 0')

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      50.0447
NDf                       =            5
Edm                       =  2.48454e-08
NCalls                    =          175
p0                        =     -17.3623   +/-   2.86855     
p1                        =      -15.577   +/-   3.72851     
p2                        =            0   +/-   5.76129     
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      40.2573
NDf                       =            5
Edm                       =  8.60482e-08
NCalls                    =          176
p0                        =     -20.9279   +/-   2.48014     
p1                        =      -17.765   +/-   3.37991     
p2                        =            0   +/-   5.85757     


In [180]:
ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 1200, 800)
lg = ROOT.TLegend(0.65, 0.75, 0.86, 0.88)

lg.AddEntry(res_graph_A, "Detector A")
lg.AddEntry(res_graph_B, "Detector B")

res_trend_A.SetLineColor(COLOR_A)
res_trend_A.SetLineStyle(2)
res_trend_A.SetLineWidth(2)
res_trend_B.SetLineColor(COLOR_B)
res_trend_B.SetLineStyle(2)
res_trend_B.SetLineWidth(2)

res_graph_A.Draw("AP")
res_graph_B.Draw("P same")
res_trend_A.Draw("same")
res_trend_B.Draw("same")
lg.Draw()

# get parameters
fit_pars_res = {}
eqboxes = {}
for counter, chip, fitfn in zip([0,1], ['A', 'B'], [res_trend_A, res_trend_B]):
    eqboxes[f'{chip}'] = ROOT.TPaveText(0.6, 0.6-counter*(0.13), 0.86, 0.8-counter*(0.13), "NDC")
    box =  eqboxes[f'{chip}']
    box.SetFillColor(0)
    box.SetBorderSize(1)
    a = fitfn.GetParameter(0)
    b = fitfn.GetParameter(1)
    c = fitfn.GetParameter(2)
    sa = fitfn.GetParError(0)
    sb = fitfn.GetParError(1)
    sc = fitfn.GetParError(2)

    fit_pars_res[f'a_{chip}'] = a
    fit_pars_res[f'b_{chip}'] = b
    fit_pars_res[f'c_{chip}'] = c
    fit_pars_res[f'sa_{chip}'] = sa
    fit_pars_res[f'sb_{chip}'] = sb
    fit_pars_res[f'sc_{chip}'] = sc

    box.AddText(f"Detector {chip}")
    box.AddText("#sqrt{a^{2} + b^{2}E + c^{2}E^{2}} / E")
    box.AddText(f"a = {a:.4f} #pm {sa:.4f}")
    box.AddText(f"b = {b:.4f} #pm {sb:.4f}")
    box.AddText(f"c = {c:.4f} #pm {sc:.4f}")
    box.Draw()

# Add labels
latex = ROOT.TLatex()
latex.SetTextSize(0.02)
latex.SetTextAlign(12)  # alineación: horizontal izquierda, vertical centrada

for i in range(len(resolutions['A'])):
    latex.DrawLatex(real_peaks_ordered[i] + 0.05, resolutions['A'][i], labels[i])  # pequeño offset en x


cv.Draw()

# LED

In [33]:
# Files
acq_nrs = [_ for _ in range(11, 23)]
led_files_A = sorted([list((project_root/'data/260519').glob(f'SUBT_A{nr}*.csv'))[0] for nr in acq_nrs])
led_files_B = sorted([list((project_root/'data/260519').glob(f'SUBT_B{nr}*.csv'))[0] for nr in acq_nrs])

hiLs = [3., 2.9, 2.8995, 2.9, 2.85, 2.84, 2.83, 2.82, 2.91, 2.93, 2.95]

led_files_B

[PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B11_Cs137_LED80ns3V.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B12_Cs137_LED80ns2,9V.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B13_Cs137_LED80ns2,8995V.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B14_Cs137_LED80ns2,9V.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B15_Cs137_LED80ns2,85V.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B16_Cs137_LED80ns2,84V.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B17_Cs137_LED80ns2,83V.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B18_Cs137_LED80ns2,82V.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B19_Cs137_LED80ns2,91V.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B20_Cs137_LED80ns2,92V.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/

In [34]:
# Acquisitions, histograms

led_acqs_A = [SiphraAcquisition(file) for file in led_files_A]
led_acqs_B = [SiphraAcquisition(file) for file in led_files_B]

led_acqs_B

[SIPHRAAcquisition(File: 'SUBT_B11_Cs137_LED80ns3V'),
 SIPHRAAcquisition(File: 'SUBT_B12_Cs137_LED80ns2,9V'),
 SIPHRAAcquisition(File: 'SUBT_B13_Cs137_LED80ns2,8995V'),
 SIPHRAAcquisition(File: 'SUBT_B14_Cs137_LED80ns2,9V'),
 SIPHRAAcquisition(File: 'SUBT_B15_Cs137_LED80ns2,85V'),
 SIPHRAAcquisition(File: 'SUBT_B16_Cs137_LED80ns2,84V'),
 SIPHRAAcquisition(File: 'SUBT_B17_Cs137_LED80ns2,83V'),
 SIPHRAAcquisition(File: 'SUBT_B18_Cs137_LED80ns2,82V'),
 SIPHRAAcquisition(File: 'SUBT_B19_Cs137_LED80ns2,91V'),
 SIPHRAAcquisition(File: 'SUBT_B20_Cs137_LED80ns2,92V'),
 SIPHRAAcquisition(File: 'SUBT_B21_Cs137_LED80ns2,93V'),
 SIPHRAAcquisition(File: 'SUBT_B22_Cs137_LED80ns2,95V')]

In [35]:
handlers

{'A_BG': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_A01_NewSourceTest_Background'), hist=<cppyy.gbl.TH1F object at 0x563dae60cd20>, src='Background', subt=None),
 'A_Cs': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_A02_NewSourceTest_Cs137'), hist=<cppyy.gbl.TH1F object at 0x563da8d79c90>, src='Cs-137', subt=<cppyy.gbl.TH1F object at 0x563da9b6e380>),
 'A_Na': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_A03_NewSourceTest_Na22'), hist=<cppyy.gbl.TH1F object at 0x563dade6c750>, src='Na-22', subt=<cppyy.gbl.TH1F object at 0x563daf342de0>),
 'A_Co': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_A04_NewSourceTest_Co60'), hist=<cppyy.gbl.TH1F object at 0x563dade1fb20>, src='Co-60', subt=<cppyy.gbl.TH1F object at 0x563db0b40f40>),
 'B_BG': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_B01_NewSourceTest_Background'), hist=<cppyy.gbl.TH1F object at 0x563dadb00bf0>, src='Background', subt=None),
 'B_Cs': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_B02_NewSourceTest_Cs137'), hist=<cppyy.g

In [36]:
CS137_MEV

[0.661]

In [37]:
1/16

0.0625